# LAB 05 — SEQUENCE MODELING: SIMPLE RNN, LSTM, AND GRU FOR SENTIMENT CLASSIFICATION
**Môn học**: Xử lý ngôn ngữ tự nhiên và ứng dụng (MAT3561) — Học kỳ I - 2026  
**Học viên thực hiện**: Vũ Tiến Đạt  
**Mã sinh viên (MSSV)**:   
**Giảng viên lý thuyết**: TS. Lê Hồng Phương  
**Giảng viên thực hành**: ThS. Phạm Ngọc Hải  


## 1. Khởi tạo môi trường & Cố định Seed Reproducibility

In [ ]:
import time
import math
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cố định random seed để đảm bảo khả năng tái lập (Reproducibility)
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')


## 14. Core implementation — Simple RNN Cell
Cài đặt lớp  từ đầu và kiểm tra tính tương đương với  của PyTorch:
58251h_t = 	anh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)58251


In [ ]:
from rnn_cell import SimpleRNNCell, verify_with_pytorch

# Chạy nghiệm thu kiểm tra tương đương
verify_with_pytorch()


## 11. Experiment 1 — Dataset Inspection (IMDb Movie Reviews)

In [ ]:
# Tải tập dữ liệu IMDb Large Movie Review Dataset
ds = load_dataset('stanfordnlp/imdb')
train_raw = list(ds['train']['text'])
train_labels = list(ds['train']['label'])
test_texts = list(ds['test']['text'])
y_test = list(ds['test']['label'])

# Chia tập huấn luyện thành Train (20,000) và Validation (5,000) có phân tầng (stratified)
train_texts, val_texts, y_train, y_val = train_test_split(
    train_raw, train_labels, test_size=5000, random_state=42, stratify=train_labels
)

print(f'Train samples: {len(train_texts)}, Val samples: {len(val_texts)}, Test samples: {len(test_texts)}')
print(f'Train class distribution: {Counter(y_train)}')
print(f'Val class distribution: {Counter(y_val)}')
print(f'Test class distribution: {Counter(y_test)}')


## 12. Tokenization và Xây dựng Vocabulary Pipeline

In [ ]:
def tokenize(text):
    return text.lower().replace('<br />', ' ').split()

vocab_counter = Counter()
for text in train_texts:
    vocab_counter.update(tokenize(text))

VOCAB_SIZE = 20000
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in vocab_counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print(f'Vocabulary size: {len(vocab)}')

def text_to_ids(text, max_len=200):
    tokens = tokenize(text)
    ids = [vocab.get(w, 1) for w in tokens[:max_len]]
    length = max(len(ids), 1)
    if len(ids) < max_len:
        ids += [0] * (max_len - len(ids))
    return ids, length

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, max_len=200):
        self.data = [text_to_ids(t, max_len) for t in texts]
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        ids, length = self.data[idx]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long), torch.tensor(length, dtype=torch.long)


## 19. Experiment 4 — Định nghĩa Kiến trúc RNN, LSTM, và GRU

In [ ]:
class SeqClassifier(nn.Module):
    def __init__(self, rnn_type, vocab_size, embed_dim=128, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn_type = rnn_type.upper()
        if self.rnn_type == 'RNN':
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif self.rnn_type == 'LSTM':
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        elif self.rnn_type == 'GRU':
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x, lengths):
        embedded = self.embedding(x)
        if self.rnn_type == 'LSTM':
            output, (h_n, c_n) = self.rnn(embedded)
        else:
            output, h_n = self.rnn(embedded)
        batch_size = x.size(0)
        idx = (lengths - 1).clamp(min=0).unsqueeze(1).unsqueeze(2).expand(batch_size, 1, output.size(2))
        last_hidden = output.gather(1, idx).squeeze(1)
        return self.fc(last_hidden)


## 22. Đánh giá Định lượng & Bảng Kết quả Thực nghiệm Tổng hợp

In [ ]:
df_results = pd.read_csv('results.csv')
display(df_results)


## 23. Đường cong học tập (Learning Curves) & Confusion Matrix

In [ ]:
from IPython.display import Image, display

print('--- Learning Curves (RNN vs LSTM vs GRU) ---')
display(Image(filename='learning_curves.png'))

print('--- Confusion Matrix (Best Model: LSTM) ---')
display(Image(filename='confusion_matrix.png'))


## 15 & 21. Minh chứng thực nghiệm: Khả năng xử lý Thứ tự từ & Phụ thuộc tầm xa

In [ ]:
with open('probe_results.json', 'r') as f:
    probes = json.load(f)
pd.DataFrame(probes)
